# JARVIS-VLA MineStudio play

Official path: `vllm serve` + `jarvisvla.evaluate.agent_wrapper.VLLM_AGENT`. Kernel **jarvis-minestudio**. Start `bash serve.sh` in another terminal unless `.rung` says `degraded`.

Recommended sampling (upstream issue #6): temperature 0.7, top_p 0.99, history_num 2.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from IPython.display import display, Markdown
from play_loop import OfficialVllmPlay, DegradedHfOneStep, is_degraded
print("degraded", is_degraded())

In [ ]:
if is_degraded():
    player = DegradedHfOneStep()
    print("HF one-step fallback — not the official serve loop")
else:
    player = OfficialVllmPlay(
        base_url="http://localhost:8000/v1",
        temperature=0.7,
        history_num=2,
        instruction_type="normal",
    )
    print("VLLM_AGENT ready")

## Simulator (optional)

Needs Xvfb or `MINESTUDIO_GPU_RENDER=1`. If this cell fails, use a saved frame below.

In [ ]:
env = None
try:
    from minestudio.simulator import MinecraftSim
    env = MinecraftSim(obs_size=(640, 360))
    obs, info = env.reset()
    frame = obs["image"] if isinstance(obs, dict) and "image" in obs else obs
    from PIL import Image
    import numpy as np
    display(Image.fromarray(np.asarray(frame)))
    print("MineStudio reset ok")
except Exception as exc:
    print("Simulator not up:", exc)
    from PIL import Image
    frame = Image.open(ROOT / "assets" / "minecraft_sample.png").convert("RGB")
    display(frame)

## Ask for an action

Reply pane is the decoded key/mouse action, not leftover English.

In [ ]:
instruction = "kill a sheep"
out = player.step(frame, instruction)
display(Markdown("**Action / reply**"))
display(Markdown(str(out.get("reply"))))
print(out)

In [ ]:
if env is not None and out.get("action") is not None:
    obs, reward, term, trunc, info = env.step(out["action"])
    frame = obs["image"] if isinstance(obs, dict) and "image" in obs else obs
    from PIL import Image
    import numpy as np
    display(Image.fromarray(np.asarray(frame)))
    print("reward", reward)
else:
    print("No env step (degraded path or no simulator).")